# Ana Agent — Colab Notebook

Simulated PT patient. Two backend options:
- **Ollama** (local, free) — runs a quantized model inside the Colab VM
- **Claude API** (Anthropic) — requires an API key

Run the cells top to bottom. Skip sections you don't need.

## 1 — Install Python dependencies

In [ ]:
!pip install -q anthropic ollama tenacity python-dotenv

## 2 — Clone the repo & set paths

If you already uploaded the repo to Drive, mount it and adjust `REPO_ROOT` instead.

In [ ]:
import os, sys

# ── Option A: clone from GitHub ──────────────────────────────────────────────
REPO_URL = "https://github.com/Tenma-Labs/Simulated-Patient-PT-chat-agent-.git"
REPO_DIR = "/content/pt-agent"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

# ── Option B: Google Drive ────────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# REPO_DIR = "/content/drive/MyDrive/pt-agent"  # adjust path

CLAUDE_VER = os.path.join(REPO_DIR, "claude_ver")
sys.path.insert(0, CLAUDE_VER)
print(f"Repo:       {REPO_DIR}")
print(f"claude_ver: {CLAUDE_VER}")

## 3 — (Ollama only) Install Ollama & pull a model

Skip this section entirely if you are using the Claude backend.

> **Recommended models for Colab free tier (T4, ~15 GB RAM)**
> - `llama3.2:3b` — fast, ~2 GB, good for quick tests
> - `llama3.2:8b` — better character fidelity, ~5 GB
> - `mistral:7b-instruct` — strong instruction-following

In [ ]:
# Install the Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, threading, time

def _start_ollama():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL)

t = threading.Thread(target=_start_ollama, daemon=True)
t.start()
time.sleep(4)   # give the server time to start
print("Ollama server started.")

In [ ]:
# Change the model tag if you want a different one
OLLAMA_MODEL = "llama3.2:3b"
!ollama pull {OLLAMA_MODEL}

## 4 — Configuration

Set your backend here. Only one section needs to be filled.

In [ ]:
# ── Choose backend ────────────────────────────────────────────────────────────
BACKEND = "ollama"          # "ollama" | "claude"

# Ollama settings (only used when BACKEND = "ollama")
OLLAMA_MODEL      = "llama3.2:3b"
OLLAMA_BASE_URL   = "http://localhost:11434"

# Claude settings (only used when BACKEND = "claude")
# Paste your key here or set the ANTHROPIC_API_KEY env var before running
ANTHROPIC_API_KEY = ""      # e.g. "sk-ant-..."
CLAUDE_MODEL      = "claude-3-5-haiku-20241022"

# Pipeline settings
GUARDRAIL         = True    # False = single-pass, faster
TEMPERATURE       = 0.7
REFRESH_TURNS     = 3       # inject constraint reminder every N turns

# ── Spec files ────────────────────────────────────────────────────────────────
# Default: looks for data/specs/ at the repo root.
# Override here if your files are elsewhere.
import os
DATA_DIR      = os.path.join(REPO_DIR, "data")
BEHAVIOR_FILE = os.path.join(DATA_DIR, "specs", "behavior.txt")
CHARACTER_FILE= os.path.join(DATA_DIR, "specs", "character.txt")
LOGS_DIR      = os.path.join(DATA_DIR, "session_logs")

os.makedirs(LOGS_DIR, exist_ok=True)
print(f"Backend:        {BACKEND}")
print(f"Model:          {OLLAMA_MODEL if BACKEND == 'ollama' else CLAUDE_MODEL}")
print(f"Guardrail:      {GUARDRAIL}")
print(f"Behavior file:  {BEHAVIOR_FILE}  exists={os.path.exists(BEHAVIOR_FILE)}")
print(f"Character file: {CHARACTER_FILE}  exists={os.path.exists(CHARACTER_FILE)}")

## 5 — Create spec files

If `behavior.txt` and `character.txt` already exist in the repo, skip this cell.
Otherwise fill in placeholders below.

In [ ]:
from pathlib import Path

specs_dir = Path(DATA_DIR) / "specs"
specs_dir.mkdir(parents=True, exist_ok=True)

# Only writes if the file is missing — won't overwrite existing content.
if not Path(BEHAVIOR_FILE).exists():
    Path(BEHAVIOR_FILE).write_text("""\
- Answer only what the student directly asks. Do not volunteer additional information.
- Use plain everyday language. Never use medical terms or diagnoses.
- Stay fully in character as the patient at all times.
- Ask at most one follow-up question per reply.
- Do not break the fourth wall or reference the simulation.
""", encoding="utf-8")
    print("Created behavior.txt (placeholder — replace with real content)")
else:
    print("behavior.txt already exists.")

if not Path(CHARACTER_FILE).exists():
    Path(CHARACTER_FILE).write_text("""\
Name: Ana Lopez
Age: 38
Occupation: Elementary school teacher
Chief complaint: Right knee pain for the past 3 weeks, started gradually.
Personality: Friendly, a little anxious about the pain, tends to ask questions back.
Background: No prior surgeries. Active lifestyle, walks 30 min daily.
""", encoding="utf-8")
    print("Created character.txt (placeholder — replace with real content)")
else:
    print("character.txt already exists.")

## 6 — Initialise provider and session

In [ ]:
from pathlib import Path
from ana_agent.providers import build_provider
from ana_agent.session import ConversationHistory

if BACKEND == "claude" and ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

MODEL = OLLAMA_MODEL if BACKEND == "ollama" else CLAUDE_MODEL

provider = build_provider(
    backend=BACKEND,
    model=MODEL,
    anthropic_api_key=os.environ.get("ANTHROPIC_API_KEY", ""),
    ollama_base_url=OLLAMA_BASE_URL,
)

history = ConversationHistory()
print(f"Provider ready: {BACKEND} / {MODEL}")

## 7 — Chat helper

Run this cell once to define the `chat()` function.

In [ ]:
import datetime as dt
from pathlib import Path
from ana_agent.pipeline import run_pipeline

_log_path = Path(LOGS_DIR) / f"colab_{BACKEND}_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

def chat(message: str) -> str:
    """Send a message to Ana and print her reply."""
    response = run_pipeline(
        user_input=message,
        history=history,
        provider=provider,
        behavior_file=Path(BEHAVIOR_FILE),
        character_file=Path(CHARACTER_FILE),
        temperature=TEMPERATURE,
        guardrail=GUARDRAIL,
        refresh_turns=REFRESH_TURNS,
    )
    history.append_user(message)
    history.append_assistant(response)

    with _log_path.open("a", encoding="utf-8") as f:
        f.write(f"USER: {message}\nANA:  {response}\n\n")

    print(f"you> {message}")
    print(f"ana> {response}")
    print(f"     [turn {history.turn_count()} | log: {_log_path.name}]")
    return response


def reset_session():
    """Clear conversation history and start a new log file."""
    global history, _log_path
    history = ConversationHistory()
    _log_path = Path(LOGS_DIR) / f"colab_{BACKEND}_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    print("Session reset.")


print("chat() and reset_session() ready.")
print(f"Log: {_log_path}")

## 8 — Talk to Ana

Call `chat("your message")` in any cell below. 
Each call appends to the running conversation history.

In [ ]:
chat("Hi Ana, I'm a PT student. Can you tell me what brought you in today?")

In [ ]:
chat("How long have you been having that pain?")

In [ ]:
chat("Does anything make it better or worse?")

In [ ]:
# Add more cells as needed, e.g.:
# chat("On a scale of 0–10, what is your pain level right now?")

## 9 — Interactive loop (optional)

Type messages one at a time. Type `exit` to stop.

In [ ]:
while True:
    user = input("you> ").strip()
    if not user:
        continue
    if user.lower() in {"exit", "quit", "/exit"}:
        print("Session ended.")
        break
    chat(user)
    print()

## 10 — Review session log

In [ ]:
print(_log_path.read_text(encoding="utf-8"))

## 11 — Reset and start a new session

In [ ]:
reset_session()